In [0]:
%pip install pytest pytest-html --quiet
dbutils.library.restartPython()

## Integration Testing Workflow for Medallion Architecture

### Setup (One-time)
1. **Create baseline fixtures**: Run your pipeline with known test data
2. **Export expected outputs**: Use Cell 2 to export Delta tables to Parquet files
3. **Move to test fixtures**: Use Cell 3 to copy Parquet files to `/Workspace/.../tests/fixtures/`

### Test Execution
1. **Load test framework**: Run Cell 30 to initialize `IntegrationTestRunner`
2. **Run tests**: Execute Cell 1 (or create custom test cells)
3. **Review results**: Check the summary output
4. **Inspect failures**: Use Cell 2 to see mismatched rows in detail

### Test Structure
Each test compares:
- **Schema**: Column names and data types must match
- **Row Count**: Same number of rows
- **Data Values**: Exact row-by-row comparison

### Key Columns
Specify primary keys for each table:
- **Bronze**: Source system keys (customer_id, product_id, order_id)
- **Silver**: Business keys after deduplication
- **Gold**: Dimension keys (for SCD Type 2, use surrogate_key + is_current)
- **Metrics**: pipeline_run_id + entity/table name

### Automation
Integrate with your DAB project:
```yaml
resources:
  jobs:
    integration_tests:
      name: "Superstore Integration Tests"
      tasks:
        - task_key: run_tests
          notebook_task:
            notebook_path: "/Users/.../tests/gold_test"
```

### Best Practices
1. **Small test datasets**: Keep fixtures small for fast tests
2. **Version fixtures**: Store in git alongside code
3. **Update on schema changes**: Re-export when pipeline logic changes
4. **Test in isolation**: Use separate catalog/schema for tests (e.g., `test_bronze`, `test_silver`)
5. **CI/CD integration**: Run tests before production deployment

In [0]:
# Use this cell to inspect failures in detail
# Run after integration tests to see which rows differ

# Example: If data comparison failed, inspect the differences
# Uncomment after running tests:

# if hasattr(tester, '_mismatches'):
#     print("Rows in expected but not in actual:")
#     tester._mismatches['expected_not_in_actual'].display()
#     
#     print("\nRows in actual but not in expected:")
#     tester._mismatches['actual_not_in_expected'].display()
# else:
#     print("No mismatches found or tests not yet run")

print("Run integration tests first, then use this cell to inspect failures")

In [0]:
# Example: Run integration tests for Bronze layer tables
# Uncomment and adjust after exporting bronze layer fixtures

# Initialize test runner
# fixtures_path = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/tests/fixtures/integration_tests"
# tester = IntegrationTestRunner(fixtures_path)

# # Test Bronze Layer Tables
# tester.run_test(
#     test_name="Bronze Customers",
#     catalog="superstore_catalog",
#     schema="dev_bronze",
#     table_name="customers",
#     layer="bronze",
#     key_columns=["customer_id"]
# )

# tester.run_test(
#     test_name="Bronze Products",
#     catalog="superstore_catalog",
#     schema="dev_bronze",
#     table_name="products",
#     layer="bronze",
#     key_columns=["product_id"]
# )

# tester.run_test(
#     test_name="Bronze Orders",
#     catalog="superstore_catalog",
#     schema="dev_bronze",
#     table_name="orders",
#     layer="bronze",
#     key_columns=["order_id"]
# )

# tester.run_test(
#     test_name="Bronze Sales",
#     catalog="superstore_catalog",
#     schema="dev_bronze",
#     table_name="sales",
#     layer="bronze",
#     key_columns=["sale_id"]  # Adjust to your actual primary key
# )

# tester.print_summary()

print("Bronze layer tests ready to run after exporting fixtures")

In [0]:
# Initialize test runner
fixtures_path = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/tests/fixtures/integration_tests"
tester = IntegrationTestRunner(fixtures_path)

# Run tests for Metrics layer (already exported)
print("Testing Metrics Layer Tables...")

# Test 1: Bronze Layer Entity Metrics
tester.run_test(
    test_name="Bronze Entity Metrics",
    catalog="superstore_catalog",
    schema="dev_metrics",
    table_name="bronze_layer_entity_metrics",
    layer="metrics",
    key_columns=["pipeline_run_id", "entity_name"]  # Adjust based on actual schema
)

# Test 2: Silver Layer Metrics
tester.run_test(
    test_name="Silver Layer Metrics",
    catalog="superstore_catalog",
    schema="dev_metrics",
    table_name="silver_layer_metrics",
    layer="metrics",
    key_columns=["pipeline_run_id", "table_name"]  # Adjust based on actual schema
)

# Test 3: Gold Layer Metrics
tester.run_test(
    test_name="Gold Layer Metrics",
    catalog="superstore_catalog",
    schema="dev_metrics",
    table_name="gold_layer_metrics",
    layer="metrics",
    key_columns=["pipeline_run_id", "table_name"]  # Adjust based on actual schema
)

# Print summary
tester.print_summary()

In [0]:
source_table = "superstore_catalog.dev_audit.products_duplicates"
destination_path = "/Volumes/workspace/default/my_filestore/integration_tests/audit/customers"

# Read the Delta table
df = spark.read.format("delta").table(source_table)

# Show what we're exporting
print(f"Exporting table: {source_table}")
print(f"Destination: {destination_path}")
print(f"Row count: {df.count()}")

# Export to Parquet
df.write.mode("overwrite").parquet(destination_path)

print(f"\n✓ Successfully exported to Parquet format")

In [0]:
# List of tables to export
tables_to_export = ["customers_dirty","products_dirty", "orders_dirty"]
catalog_schema = "superstore_catalog.dev_quarantine"
base_path = "/Volumes/workspace/default/my_filestore/integration_tests/quarantine"

def export_table_to_parquet(table_name, catalog_schema, output_base_path):
    """Export a single Delta table to a named Parquet file"""
    source = f"{catalog_schema}.{table_name}"
    temp_path = f"{output_base_path}/_temp_{table_name}"
    final_path = f"{output_base_path}/{table_name}.parquet"
    
    print(f"\n{'='*60}")
    print(f"Processing: {source}")
    
    # Read Delta table
    df = spark.read.format("delta").table(source)
    row_count = df.count()
    print(f"Rows: {row_count}")
    
    # Write as single partition to temp location
    df.coalesce(1).write.mode("overwrite").parquet(temp_path)
    
    # Find the part file
    files = dbutils.fs.ls(temp_path)
    part_file = [f for f in files if f.name.endswith('.parquet')][0]
    
    # Remove old file if exists
    try:
        dbutils.fs.rm(final_path)
    except:
        pass
    
    # Move to final location with clean name
    dbutils.fs.mv(part_file.path, final_path)
    
    # Clean up temp directory
    dbutils.fs.rm(temp_path, recurse=True)
    
    print(f"✓ Exported to: {final_path}")
    return row_count

# Export all tables
print(f"Starting export of {len(tables_to_export)} tables...")
results = {}

for table in tables_to_export:
    try:
        count = export_table_to_parquet(table, catalog_schema, base_path)
        results[table] = {"status": "success", "rows": count}
    except Exception as e:
        results[table] = {"status": "failed", "error": str(e)}
        print(f"✗ Failed: {e}")

# Summary
print(f"\n{'='*60}")
print("EXPORT SUMMARY")
print(f"{'='*60}")
for table, result in results.items():
    if result["status"] == "success":
        print(f"✓ {table:15s} - {result['rows']:,} rows")
    else:
        print(f"✗ {table:15s} - FAILED: {result['error']}")


In [0]:
# Source and destination paths
source_base = "/Volumes/workspace/default/my_filestore/integration_tests"
dest_base = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/tests/fixtures/integration_tests"

# Create destination directory if it doesn't exist
dbutils.fs.mkdirs(f"{dest_base}/silver")

# List all parquet files in the source metrics directory
print("Moving files from volume to workspace...\n")

try:
    # Get list of files in source
    source_files = dbutils.fs.ls(f"{source_base}/quarantine")
    
    moved_count = 0
    for file_info in source_files:
        if file_info.name.endswith('.parquet'):
            source_file = file_info.path
            dest_file = f"{dest_base}/silver/{file_info.name}"
            
            print(f"Moving: {file_info.name}")
            print(f"  From: {source_file}")
            print(f"  To: {dest_file}")
            
            # Copy file to workspace location
            dbutils.fs.cp(source_file, dest_file, recurse=True)
            moved_count += 1
            print(f"  ✓ Moved successfully\n")
    
    print(f"\n{'='*60}")
    print(f"✓ Successfully moved {moved_count} files")
    print(f"✓ Destination: {dest_base}/silver")
    
    # List files in destination to verify
    print(f"\nFiles in destination:")
    dest_files = dbutils.fs.ls(f"{dest_base}/silver")
    for f in dest_files:
        print(f"  - {f.name} ({f.size} bytes)")
        
except Exception as e:
    print(f"Error: {e}")

In [0]:
spark.read.parquet("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/tests/fixtures/integration_tests/gold/dim_products.parquet").display()

In [0]:
gold_tbl_customers= 'superstore_catalog.superstore_gold.dim_customers'

In [0]:
# One-current-row rule (MOST IMPORTANT)
# Each customer must have exactly ONE current row
spark.sql(f"""
SELECT customer_id, COUNT(*) cnt
FROM superstore_catalog.superstore_gold.dim_customers
WHERE is_current = true
GROUP BY customer_id
HAVING cnt != 1
""").show()

In [0]:
# SCD2 timeline correctness- No overlapping date ranges
spark.sql(f"""
SELECT customer_id
FROM superstore_catalog.superstore_gold.dim_customers
GROUP BY customer_id
HAVING COUNT(*) > 1
AND MAX(effective_to) IS NOT NULL
AND MIN(effective_from) > MAX(effective_to)
""").show()


In [0]:
# Late-arriving data validation- Pick one customer_id:
# Check:
# Older change appears before
# Newer record is is_current = true
# Previous record has effective_to

spark.sql(f"""
SELECT *
FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880'
ORDER BY effective_from
""").display()


In [0]:
# Idempotency test (CRITICAL)
# Run the full notebook TWICE
# Then check:Should return ZERO rows

spark.sql(f"""
SELECT customer_id, effective_from, COUNT(*)
FROM superstore_catalog.superstore_gold.dim_customers
GROUP BY customer_id, effective_from
HAVING COUNT(*) > 1
""").display()


In [0]:
# Soft delete validation-Delete a customer from Silver:
spark.sql("""
DELETE FROM superstore_catalog.superstore_silver.customers
WHERE customer_id = 'PR-18880'
""")


Run pipeline again, then:is_current = false
effective_to populated

In [0]:
spark.sql(f"""
SELECT *
FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880'
""").display()


In [0]:
# Hash change validation-Update a Silver attribute:

In [0]:
spark.sql("""
UPDATE superstore_catalog.superstore_silver.customers
SET city = 'NEW CITY'
WHERE customer_id = 'PR-18880'
""")


Run pipeline again:

In [0]:
spark.sql(f"""
SELECT customer_id, city, is_current
FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880'
ORDER BY effective_from
""").display()

# Old record expired
# New record current

In [0]:
spark.table('superstore_catalog.superstore_gold.dim_customers').count()

In [0]:
spark.table('superstore_catalog.superstore_silver.customers').count()

In [0]:
# Delete rows where customer_id=90 AND city='oo'
spark.sql("""
DELETE FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880' AND city = 'NEW CITY'
""")



In [0]:
spark.sql("""
update superstore_catalog.superstore_silver.customers
set city='Oakland'
where customer_id= 'PR-18880'
""")



In [0]:
spark.sql("""
select city 
from superstore_catalog.superstore_silver.customers
where customer_id= 'PR-18880'
""").display()

In [0]:
spark.sql(f"""
SELECT customer_id, city, is_current
FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880'
ORDER BY effective_from
""").display()

# Old record expired
# New record current

In [0]:
spark.table('superstore_catalog.superstore_gold.dim_customers').count()

In [0]:
spark.table('superstore_catalog.superstore_silver.customers').count()

In [0]:
gold_df = spark.table("superstore_catalog.superstore_gold.dim_customers")
gold_df.count()  # total historical + current rows
gold_df.filter(col("is_current") == True).count()  # current only


In [0]:
from pyspark.sql.functions import col
spark.table('superstore_catalog.superstore_gold.dim_customers')\
    .filter(col('customer_id')=='PR-18880').display()

In [0]:
from pyspark.sql.functions import col

silver_df = spark.table("superstore_catalog.superstore_silver.customers")
gold_df = spark.table("superstore_catalog.superstore_gold.dim_customers")

# Rows in Silver not in Gold (by customer_id) at all
missing_in_gold = silver_df.join(gold_df, on="customer_id", how="left_anti")
missing_in_gold.display()

# Rows in Silver where Gold has is_current = false
expired_in_gold = silver_df.join(
    gold_df.filter(col("is_current") == False),
    on="customer_id",
    how="left_semi"
)
expired_in_gold.display()


In [0]:
spark.sql(f"""
SELECT *
FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880'
""").display()

In [0]:
spark.sql(f"""
SELECT *
FROM superstore_catalog.superstore_silver.customers
WHERE customer_id = 'PR-18880'
""").display()

In [0]:
# Integration Test Setup
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, when
from typing import Dict, List, Tuple

class IntegrationTestRunner:
    """Framework for comparing actual Delta tables against expected Parquet fixtures"""
    
    def __init__(self, fixtures_base_path: str):
        self.fixtures_base = fixtures_base_path
        self.test_results = []
    
    def load_expected(self, layer: str, table_name: str) -> DataFrame:
        """Load expected data from Parquet fixtures"""
        path = f"{self.fixtures_base}/{layer}/{table_name}.parquet"
        return spark.read.parquet(path)
    
    def load_actual(self, catalog: str, schema: str, table_name: str) -> DataFrame:
        """Load actual data from Delta table"""
        return spark.table(f"{catalog}.{schema}.{table_name}")
    
    def compare_schemas(self, expected: DataFrame, actual: DataFrame, test_name: str) -> bool:
        """Compare schemas between expected and actual"""
        expected_schema = set((f.name, f.dataType.simpleString()) for f in expected.schema.fields)
        actual_schema = set((f.name, f.dataType.simpleString()) for f in actual.schema.fields)
        
        if expected_schema == actual_schema:
            self.test_results.append({"test": test_name, "check": "Schema", "status": "PASS"})
            return True
        else:
            missing = expected_schema - actual_schema
            extra = actual_schema - expected_schema
            self.test_results.append({
                "test": test_name, 
                "check": "Schema", 
                "status": "FAIL",
                "details": f"Missing: {missing}, Extra: {extra}"
            })
            return False
    
    def compare_row_counts(self, expected: DataFrame, actual: DataFrame, test_name: str) -> bool:
        """Compare row counts"""
        expected_count = expected.count()
        actual_count = actual.count()
        
        if expected_count == actual_count:
            self.test_results.append({
                "test": test_name, 
                "check": "Row Count", 
                "status": "PASS",
                "details": f"{actual_count} rows"
            })
            return True
        else:
            self.test_results.append({
                "test": test_name, 
                "check": "Row Count", 
                "status": "FAIL",
                "details": f"Expected: {expected_count}, Actual: {actual_count}"
            })
            return False
    
    def compare_data(self, expected: DataFrame, actual: DataFrame, test_name: str, key_columns: List[str]) -> bool:
        """Compare actual data values"""
        # Sort both dataframes by key columns for comparison
        expected_sorted = expected.orderBy(*key_columns)
        actual_sorted = actual.orderBy(*key_columns)
        
        # Find differences
        expected_not_in_actual = expected_sorted.subtract(actual_sorted)
        actual_not_in_expected = actual_sorted.subtract(expected_sorted)
        
        diff_expected = expected_not_in_actual.count()
        diff_actual = actual_not_in_expected.count()
        
        if diff_expected == 0 and diff_actual == 0:
            self.test_results.append({
                "test": test_name, 
                "check": "Data Match", 
                "status": "PASS"
            })
            return True
        else:
            self.test_results.append({
                "test": test_name, 
                "check": "Data Match", 
                "status": "FAIL",
                "details": f"Rows only in expected: {diff_expected}, only in actual: {diff_actual}"
            })
            # Store mismatches for inspection
            self._mismatches = {
                "expected_not_in_actual": expected_not_in_actual,
                "actual_not_in_expected": actual_not_in_expected
            }
            return False
    
    def run_test(self, test_name: str, catalog: str, schema: str, table_name: str, 
                 layer: str, key_columns: List[str]) -> Dict:
        """Run full integration test for a table"""
        print(f"\n{'='*70}")
        print(f"Running Test: {test_name}")
        print(f"{'='*70}")
        
        # Load data
        expected = self.load_expected(layer, table_name)
        actual = self.load_actual(catalog, schema, table_name)
        
        # Run checks
        schema_pass = self.compare_schemas(expected, actual, test_name)
        count_pass = self.compare_row_counts(expected, actual, test_name)
        data_pass = self.compare_data(expected, actual, test_name, key_columns)
        
        overall_status = "PASS" if (schema_pass and count_pass and data_pass) else "FAIL"
        
        return {
            "test": test_name,
            "overall": overall_status,
            "schema": schema_pass,
            "count": count_pass,
            "data": data_pass
        }
    
    def print_summary(self):
        """Print test results summary"""
        print(f"\n\n{'='*70}")
        print("INTEGRATION TEST SUMMARY")
        print(f"{'='*70}")
        
        for result in self.test_results:
            status_symbol = "✓" if result["status"] == "PASS" else "✗"
            details = f" - {result.get('details', '')}" if "details" in result else ""
            print(f"{status_symbol} {result['test']:30s} | {result['check']:15s} | {result['status']}{details}")
        
        total = len(self.test_results)
        passed = sum(1 for r in self.test_results if r["status"] == "PASS")
        print(f"\n{passed}/{total} checks passed")

print("✓ Integration test framework loaded")

#products-test

In [0]:
# One-current-row rule (MOST IMPORTANT)
# Each customer must have exactly ONE current row
spark.sql(f"""
SELECT product_id, COUNT(*) cnt
FROM superstore_catalog.superstore_gold.dim_products
WHERE is_current = true
GROUP BY product_id
HAVING cnt != 1
""").show()

In [0]:
silver_tbl = "superstore_catalog.superstore_silver.products"
spark.table(silver_tbl).count()

In [0]:
spark.table('superstore_catalog.superstore_gold.dim_products').count()

In [0]:
spark.table('superstore_catalog.superstore_gold.dim_products').count()

In [0]:
spark.table('superstore_catalog.superstore_gold.dim_products').count()

In [0]:
silver_count = spark.table("superstore_catalog.superstore_silver.products").count()
gold_count = spark.table("superstore_catalog.superstore_gold.dim_products").count()

print(f"Silver: {silver_count}, Gold: {gold_count}")


In [0]:
silver_ids = spark.table("superstore_catalog.superstore_silver.products").select("product_id")
gold_ids = spark.table("superstore_catalog.superstore_gold.dim_products").select("product_id")
missing_in_gold = silver_ids.join(gold_ids, on="product_id", how="left_anti")
missing_in_gold.show()


facts_orders

In [0]:
#count silver and gold rows
silver_count = spark.table("superstore_catalog.superstore_silver.orders").count()
gold_count = spark.table("superstore_catalog.superstore_gold.facts_orders").count()

print(f"Silver: {silver_count}, Gold: {gold_count}")


In [0]:
# Check recent load timestamps
from pyspark.sql.functions import col
spark.table("superstore_catalog.superstore_gold.facts_orders").orderBy(col("load_ts").desc()).limit(10).display()



In [0]:
# Compare hashes
from pyspark.sql.functions import col
gold_df = spark.table("superstore_catalog.superstore_gold.facts_orders").alias("gold")
silver_df = spark.table("superstore_catalog.superstore_silver.orders").alias("silver")

gold_df.join(silver_df, on="order_id") \
    .filter(col("gold.silver_order_hash_id") != col("silver.silver_order_hash_id")) \
    .display()



sales


In [0]:
silver_tbl = "superstore_catalog.superstore_silver.sales"
spark.table(silver_tbl).count()

In [0]:
silver_tbl = "superstore_catalog.superstore_gold.facts_sales"
spark.table(silver_tbl).count()

In [0]:
# scd2prepared

In [0]:
gold_customers_scd_df.display()

In [0]:
gold_customers_df .display()